Using the Stage 1 to extract the Transmission Map and Atmospheric Light

In [12]:
import sys 

sys.path.append("..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import torch.backends.cudnn as cudnn
import random

import os
os.chdir("/workspace/dehazing")

In [2]:
def set_seed(seed):
    """Sets the seed for reproducibility across random, numpy, and torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        cudnn.deterministic = True
        cudnn.benchmark = False

### Auxiliary Code

In [3]:

def get_pad_layer(pad_type):
    if(pad_type in ['refl','reflect']):
        PadLayer = nn.ReflectionPad2d
    elif(pad_type in ['repl','replicate']):
        PadLayer = nn.ReplicationPad2da
    elif(pad_type=='zero'):
        PadLayer = nn.ZeroPad2d
    else:
        print(f'Pad type [{pad_type}] not recognized')
    return PadLayer


class AntiAlias_Downsample(nn.Module):
    def __init__(self, channels, pad_type = 'reflect', filt_size = 3, 
                        stride = 2, pad_off = 0):
        super(AntiAlias_Downsample, self).__init__()
        self.filt_size = filt_size
        self.pad_off = pad_off
        self.pad_type = pad_type

        # Asymmetric padding (round up at the top and round down at the bottom)
        # Perfect when kernel size is 2 
        self.pad_sizes = [int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2)),
                          int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2))]
        self.pad_sizes = [pad_size + pad_off for pad_size in self.pad_sizes]
        self.stride = stride 
        self.off = int((self.stride - 1) / 2.)
        self.channels = channels 

        # Define the binomial filter weights
        if(self.filt_size==1):
            a = np.array([1.,])
        elif(self.filt_size==2):
            a = np.array([1., 1.])
        elif(self.filt_size==3):
            a = np.array([1., 2., 1.])
        elif(self.filt_size==4):    
            a = np.array([1., 3., 3., 1.])
        elif(self.filt_size==5):    
            a = np.array([1., 4., 6., 4., 1.])
        elif(self.filt_size==6):    
            a = np.array([1., 5., 10., 10., 5., 1.])
        elif(self.filt_size==7):    
            a = np.array([1., 6., 15., 20., 15., 6., 1.])
            
        # Create a 2D filter by taking the outer product of the 1D filter
        filt = torch.tensor(a[:, None] * a[None, :], dtype = torch.float32)
        filt = filt / torch.sum(filt) # Normalize

        # Reshape to (out_channels, in_channels/groups, kH, kW) for 
        # depthwise convolution
        filt = filt.view(1, 1, filt_size, filt_size)
        filt = filt.repeat(channels, 1, 1, 1)

        # Register as a buffer so PyTorch knows these are NOT trainable parameters
        self.register_buffer('filt', filt)
        self.pad = get_pad_layer(pad_type)(self.pad_sizes)

    def forward(self, inp):
        if (self.filt_size == 1):
            if (self.pad_off == 0):
                return inp[:, :, ::self.stride, ::self.stride] 
            else:
                return self.pad(inp)[:, :, ::self.stride, ::self.stride] 

        else:
            return F.conv2d(self.pad(inp), self.filt, stride = self.stride, groups = inp.shape[1])

### Official Implementations

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [5]:
class BilinearUpsample(nn.Module):
    """
    Used by BOTH variants. Guarantees no checkerboard artifacts during decoding.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        return self.conv(self.up(x))

In [6]:
class VariantA_StandardDownsample(nn.Module):
    """
    Standard stride=2 convolution. 
    Prone to aliasing and shift-variance (ignores Nyquist theorem).
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # Strided convolution drops 75% of pixels abruptly
        self.down = nn.Conv2d(dim_in, dim_out, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        return self.down(x)

In [7]:
class VariantB_AntiAliasedDownsample(nn.Module):
    """
    Anti-Aliased Downsampling (BlurPool) based on Richard Zhang's paper.
    Preserves shift-invariance and prevents high-frequency aliasing.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # 1. Feature Mixing (Stride 1 preserves all spatial information)
        self.conv = nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1)
        # 2. Anti-aliased spatial reduction (Low-pass filter + subsampling)
        # NOTE: Make sure your AntiAlias_Downsample class is defined in the script!
        self.aa_down = AntiAlias_Downsample(channels=dim_out, filt_size=3, stride=2)

    def forward(self, x):
        return self.aa_down(self.conv(x))

In [8]:
# -----------------------------------------------------------------
# 3. THE ABLATION MODEL
# -----------------------------------------------------------------
class AblationPhysicsEstimator(nn.Module):
    """
    Stage 1: Configurable Physics Estimator.
    Tests Standard Downsampling vs Anti-Aliased Downsampling.
    Both use Bilinear Upsampling to isolate the effect.
    """
    def __init__(self, variant='B', in_channels=3, base_dim=32):
        super().__init__()
        self.variant = variant
        
        # --- INITIAL ENCODER ---
        self.init_conv = nn.Conv2d(in_channels, base_dim, kernel_size=3, padding=1)
        self.enc1 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)
        
        # ==========================================
        # --- ABLATION SWITCH LOGIC (DOWNSAMPLING)
        # ==========================================
        if self.variant == 'A':
            print("Initializing Stage 1 with Variant A (Standard Stride=2 Downsampling)")
            self.down1 = VariantA_StandardDownsample(base_dim, base_dim * 2)
            self.down2 = VariantA_StandardDownsample(base_dim * 2, base_dim * 4)
        elif self.variant == 'B':
            print("Initializing Stage 1 with Variant B (Anti-Aliased Downsampling)")
            self.down1 = VariantB_AntiAliasedDownsample(base_dim, base_dim * 2)
            self.down2 = VariantB_AntiAliasedDownsample(base_dim * 2, base_dim * 4)
        else:
            raise ValueError("Variant must be 'A' or 'B'")
            
        self.enc2 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)
        
        # --- BOTTLENECK ---
        self.bottleneck = nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=1)
        
        # --- GLOBAL ATMOSPHERIC LIGHT (A) HEAD ---
        self.A_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(base_dim * 4, 3), nn.Sigmoid() 
        )
        
        # --- DECODER (SHARED: Bilinear Upsampling for BOTH variants) ---
        self.up1 = BilinearUpsample(base_dim * 4, base_dim * 2)
        self.dec1_conv = nn.Conv2d(base_dim * 4, base_dim * 2, kernel_size=1) 
        self.dec1 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)
        
        self.up2 = BilinearUpsample(base_dim * 2, base_dim)
        self.dec2_conv = nn.Conv2d(base_dim * 2, base_dim, kernel_size=1)
        self.dec2 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)
        
        # --- TRANSMISSION MAP (t) HEAD ---
        self.t_head = nn.Sequential(
            nn.Conv2d(base_dim, 1, kernel_size=3, padding=1),
            nn.Sigmoid() 
        )

    def forward(self, x):
        # Encode
        e1 = self.enc1(self.init_conv(x))
        d1 = self.down1(e1)
        
        e2 = self.enc2(d1)
        d2 = self.down2(e2)
        
        # Bottleneck
        b = self.bottleneck(d2)
        A = self.A_head(b).view(-1, 3, 1, 1)
        
        # Decode
        u1 = self.up1(b)
        u1 = torch.cat([u1, e2], dim=1) # Skip connection
        u1 = self.dec1(self.dec1_conv(u1))
        
        u2 = self.up2(u1)
        u2 = torch.cat([u2, e1], dim=1) # Skip connection
        u2 = self.dec2(self.dec2_conv(u2))
        
        t_map = self.t_head(u2)
        return t_map, A

In [15]:
from data.utils import get_haze_transforms, partition_dataset
from torch.utils.data import Subset
from losses import CharbonnierLoss

In [18]:
import torch
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset
from torchvision.transforms import v2


class RESIDE_Indoor(Dataset):
    def __init__(self, dataset_path, transform=None):
        self.root_dir = Path(dataset_path)
        self.metadata_csv = pd.read_csv(self.root_dir / "metadata.csv")

        self.transform = transform
        self.data = []
        
        for idx, row in self.metadata_csv.iterrows():
            clean_path = self.root_dir / row["clear_image_path"]
            hazy_paths_str = row["hazy_image_paths"]
            hazy_image_paths = [
                path.strip()
                for path in hazy_paths_str.strip("[]").replace("'", "").split(",")
            ]
            list_hazy_paths = [
                self.root_dir / hazy_path for hazy_path in hazy_image_paths
            ]
            
            for hazy_path in list_hazy_paths:
                # --- NEW LOGIC: Deduce the Transmission Map Path ---
                # Example: hazy_path.name is "1_1_0.90179.png"
                hazy_filename = hazy_path.name
                parts = hazy_filename.split('_')
                
                # Reconstruct trans filename: "1_1.png"
                if len(parts) >= 2:
                    trans_filename = f"{parts[0]}_{parts[1]}.png"
                else:
                    trans_filename = hazy_filename # Fallback just in case
                
                trans_path = self.root_dir / "trans" / trans_filename
                # ---------------------------------------------------

                data_item = {
                    "index": idx, 
                    "clean": clean_path, 
                    "hazy": hazy_path,
                    "trans": trans_path # Store the trans path
                }

                self.data.append(data_item)

    def __repr__(self):
        return "RESIDE Indoor"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_item = self.data[idx]
        clean_path = data_item["clean"]
        hazy_path = data_item["hazy"]
        trans_path = data_item["trans"]

        try:
            clean_img = Image.open(clean_path).convert("RGB")
            hazy_img = Image.open(hazy_path).convert("RGB")
            # Load transmission map as Grayscale ("L")
            trans_img = Image.open(trans_path).convert("L") 
        except FileNotFoundError:
            print(f"Error: Missing image file at {clean_path}, {hazy_path}, or {trans_path}. Skipping")
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            # IMPORTANT WARNING: 
            # If your 'get_haze_transforms' function only expects 2 inputs, 
            # you must update it to accept and return 3 inputs!
            clean_img, hazy_img, trans_img = self.transform(clean_img, hazy_img, trans_img)
        else:
            # Fallback tensorization
            clean_img = (
                torch.as_tensor(np.array(clean_img)).permute(2, 0, 1).float() / 255.0
            )
            hazy_img = (
                torch.as_tensor(np.array(hazy_img)).permute(2, 0, 1).float() / 255.0
            )
            # Add channel dimension to grayscale image (H, W) -> (1, H, W)
            trans_img = (
                torch.as_tensor(np.array(trans_img)).unsqueeze(0).float() / 255.0
            )

        return hazy_img, clean_img, trans_img

In [19]:
def get_reside_indoor_transforms(resize_size: int = 256):
    """
    Returns both train and val transforms strictly tuned for RESIDE-INDOOR.
    Safely handles the optional 3rd 'trans' (transmission) map.
    """
    
    # 1. Base Formatting: Convert to tensor [0, 1] -> Normalize to [-1, 1]
    to_tensor_norm = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])

    # 2. Geometric Sync: Applied equally to clean, hazy, and trans to preserve alignment
    geometric_sync = v2.Compose([
        v2.RandomCrop(resize_size, pad_if_needed=True),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.5),
    ])

    # 3. Color Jitter: Tuned specifically for RESIDE-INDOOR lighting
    color_jitter = v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.01)

    # --- TRAIN TRANSFORM ---
    def train_transform(clean, hazy, trans=None):
        # A. Apply spatial changes synchronously
        if trans is not None:
            clean, hazy, trans = geometric_sync(clean, hazy, trans)
        else:
            clean, hazy = geometric_sync(clean, hazy)

        # B. Apply color distortion ONLY to the hazy image
        hazy = color_jitter(hazy)

        # C. Tensor & Normalization
        clean, hazy = to_tensor_norm(clean), to_tensor_norm(hazy)
        
        if trans is not None:
            # Trans stays [0, 1] for physics math, NO normalization!
            trans = v2.functional.to_dtype(v2.functional.to_image(trans), torch.float32, scale=True)
            return clean, hazy, trans
            
        return clean, hazy

    # --- VAL TRANSFORM ---
    def val_transform(clean, hazy, trans=None):
        # Validation just normalizes. NO cropping or flipping.
        clean, hazy = to_tensor_norm(clean), to_tensor_norm(hazy)
        
        if trans is not None:
            trans = v2.functional.to_dtype(v2.functional.to_image(trans), torch.float32, scale=True)
            return clean, hazy, trans
            
        return clean, hazy

    return train_transform, val_transform

In [20]:
set_seed(42)

resolution = 256
verbose = True
num_subset_samples = 500

train_transform, val_transform = get_reside_indoor_transforms(resize_size = resolution)

data_path = "dataset/indoor-training-set/"
dataset = RESIDE_Indoor(dataset_path=data_path)

indices = torch.randperm(len(dataset))[:num_subset_samples].tolist()
subset_dataset = Subset(dataset, indices)

train_dataset, val_dataset = partition_dataset(
    subset_dataset,
    train_transform, 
    val_transform,
    train_ratio = 0.8
)

print(f"Total Subset Size: {len(subset_dataset)}")
print(f"Training Set Size: {len(train_dataset)}")
print(f"Validation Set Size: {len(val_dataset)}")

Total Subset Size: 500
Training Set Size: 400
Validation Set Size: 100


### Testing 

If we use the Self-Supervised Setup (when there is no transmission light, 
we need to use the TV Loss since there is not depth map from scratch and also the Atmospheric Prior). Since in the synthetic dataset, it already has though transmission map so we can drop those 

In [16]:
class Stage1_RESIDELoss(nn.Module):
    """
    Supervised Physics Loss for Stage 1 (Using RESIDE Ground Truths).
    Forces exact t(x) matching, and uses reconstruction to derive A.
    """
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()
        self.charbonnier = CharbonnierLoss()
        
        # We weigh the direct supervision heavily to ensure the geometry is perfect
        self.w_t = 10.0
        self.w_recon = 1.0

    def forward(self, pred_t_map, pred_A, gt_t_map, clean_img_01, hazy_img_01):
        # 1. Clamp for safety
        t_map_safe = torch.clamp(pred_t_map, min=0.01, max=1.0)
        pred_A_safe = torch.clamp(pred_A, min=0.0, max=1.0)
        
        # 2. Direct Supervision on t(x) (The most important part!)
        loss_t = self.mse(t_map_safe, gt_t_map)
        
        # 3. Physical Reconstruction (Forces the network to figure out A)
        I_recon = clean_img_01 * t_map_safe + pred_A_safe * (1.0 - t_map_safe)
        loss_recon = self.charbonnier(I_recon, hazy_img_01)
        
        # 4. Total Loss
        total_loss = (self.w_t * loss_t) + (self.w_recon * loss_recon)
        
        # 5. Return dict for TensorBoard tracking
        return total_loss, {
            "Stage1_RESIDE/Total_Loss": total_loss.item(),
            "Stage1_RESIDE/Transmission_MSE": loss_t.item(),
            "Stage1_RESIDE/Reconstruction": loss_recon.item(),
            "Physics/Mean_Predicted_T": t_map_safe.mean().item(),
            "Physics/Mean_GT_T": gt_t_map.mean().item(),
            "Physics/Mean_Predicted_A": pred_A_safe.mean().item()
        }

In [ ]:
# class Stage1_PhysicsLoss(nn.Module):
#     """
#     Self-Supervised Physics Loss for Stage 1.
#     Uses the Clear Image (J) and predicted (t, A) to reconstruct the Hazy Image (I).
#     """
#     def __init__(self):
#         super().__init__()
#         self.charbonnier = CharbonnierLoss() # L1 with epsilon for stability
        
#         # Weights for Stage 1
#         self.w_recon = 1.0
#         self.w_tv = 0.05   # Keep small so it doesn't over-blur t(x) edges
#         self.w_atm = 0.01  # Gentle push to keep A reasonable

#     def get_gradients(self, img):
#         dy = img[:, :, 1:, :] - img[:, :, :-1, :]
#         dx = img[:, :, :, 1:] - img[:, :, :, :-1]
#         return dy, dx
    
#     def forward(self, pred_t_map, pred_A, clean_img_01, hazy_img_01):
#         # 1. Clamp predictions for physical safety
#         t_map_safe = torch.clamp(pred_t_map, min=0.01, max=1.0)
#         pred_A_safe = torch.clamp(pred_A, min=0.0, max=1.0)
        
#         # 2. Physics Reconstruction Loss (The core of Stage 1)
#         # Equation: I = J * t + A * (1 - t)
#         I_reconstructed = clean_img_01 * t_map_safe + pred_A_safe * (1.0 - t_map_safe)
#         loss_recon = self.charbonnier(I_reconstructed, hazy_img_01)

#         # 3. Total Variation Regularization (Smoothness for depth map)
#         dy, dx = self.get_gradients(t_map_safe)
#         loss_tv = torch.mean(torch.abs(dy)) + torch.mean(torch.abs(dx))

#         # 4. Atmospheric Regularizer (Push A away from pure black)
#         loss_atm = torch.mean(F.relu(0.05 - pred_A_safe)) 

#         # Aggregate
#         total_loss = (self.w_recon * loss_recon) + \
#                      (self.w_tv * loss_tv) + \
#                      (self.w_atm * loss_atm)

#         # Return dict for TensorBoard
#         return total_loss, {
#             "Stage1/Total_Loss": total_loss.item(),
#             "Stage1/Reconstruction": loss_recon.item(),
#             "Stage1/TV_Smoothness": loss_tv.item(),
#             "Stage1/Atm_Prior": loss_atm.item(),
#             "Physics/Mean_T": t_map_safe.mean().item(), # Track average haze density
#             "Physics/Mean_A": pred_A_safe.mean().item()
#         }

In [ ]:
import os
import torchvision
from torch.utils.tensorboard import SummaryWriter

# --- 1. INITIALIZE TENSORBOARD ---
# Create separate folders for your variants so you can compare them side-by-side!
variant_name = "Variant_B_AntiAliased" # Change to "Variant_A_Standard" when testing A
log_dir = os.path.join("runs", "Stage1_Ablation", variant_name)
writer = SummaryWriter(log_dir=log_dir)

# Initialize your model and loss
model = AblationPhysicsEstimator(variant='B').to(device)
criterion = Stage1_PhysicsLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# --- 2. INSIDE YOUR TRAINING LOOP ---
global_step = 0

for epoch in range(num_epochs):
    for batch_idx, (hazy_img, clean_img) in enumerate(dataloader):
        hazy_img = hazy_img.to(device)
        clean_img = clean_img.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        pred_t_map, pred_A = model(hazy_img)
        
        # Calculate loss
        loss, loss_dict = criterion(pred_t_map, pred_A, clean_img, hazy_img)
        
        loss.backward()
        optimizer.step()
        
        # --- 3. LOG SCALARS (Metrics) ---
        if global_step % 10 == 0: # Log every 10 steps
            for key, value in loss_dict.items():
                writer.add_scalar(key, value, global_step)
        
        # --- 4. LOG IMAGES (Visual Debugging) ---
        # Do this rarely (e.g., once an epoch or every 500 steps) to save memory
        if global_step % 500 == 0:
            with torch.no_grad():
                # Reconstruct image just for visualization
                I_recon = clean_img * pred_t_map + pred_A * (1.0 - pred_t_map)
                
                # Create a grid: [Hazy Input | Clean GT | Predicted T_Map | Reconstructed Hazy]
                # t_map is 1-channel, so we repeat it to 3 channels for the grid
                t_map_vis = pred_t_map.repeat(1, 3, 1, 1) 
                
                # Grab just the first 4 images of the batch to keep the grid clean
                img_grid = torchvision.utils.make_grid(
                    torch.cat([
                        hazy_img[:4], 
                        clean_img[:4], 
                        t_map_vis[:4], 
                        I_recon[:4]
                    ], dim=0), 
                    nrow=4, normalize=True
                )
                
                writer.add_image('Stage1_Visuals/Hazy_Clean_Tmap_Recon', img_grid, global_step)
                
        global_step += 1

# --- 5. CLOSE TENSORBOARD AT THE END ---
writer.close()